In [ ]:
# Installation des dépendances
!pip install datasets pandas sentencepiece

from datasets import load_dataset
import pandas as pd

ds = load_dataset("eliezermga/ruund-french-parallel-corpus")
df = pd.DataFrame(ds['train'])
print(f"Taille initiale du corpus : {len(df)} paires.")
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

ruund-french.tsv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/8400 [00:00<?, ? examples/s]

Taille initiale du corpus : 8400 paires.


,Ruund,French
0,"Aam nid kwaam nkay chilil mwaan, chikweeza kum...",Je ne suis quant à moi qu’une simple antilope ...
1,"Aay aana, aay iin twuvum","D’un côté vous avez des enfants chéris, de l’a..."
2,"Adaap ngendj, mwiiningand madyaaj mend","Le visiteur est toujours le mieux servi, n’en ..."
3,"Adangang kwend, piikil anch afangang kwend",Mieux vaut souhaiter que le prochain vive le p...
4,"Akandiling payaaw, piikil anch ezakw kal ni mash",Mieux vaut prévenir que guérir


**Normalisation Unicode et Nettoyage**

---


Le **ruund** étant une langue bantoue à
tons, il est crucial d'utiliser la normalisation NFD pour séparer les caractères de base de leurs signes diacritiques avant le traitement
Cette étape élimine également le bruit (ponctuation excessive, caractères non-ASCII)


In [2]:
import unicodedata
import re
import string

def preprocess_text(text):
    if not isinstance(text, str): return ""
    # Normalisation NFD pour gérer les marqueurs de tons bantous
    text = unicodedata.normalize('NFD', text)
    # Conversion en minuscules
    text = text.lower()
    # Suppression de la ponctuation
    table = str.maketrans('', '', string.punctuation)
    text = text.translate(table)
    # Suppression des espaces multiples
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['Ruund'] = df['Ruund'].apply(preprocess_text)
df['French'] = df['French'].apply(preprocess_text)

**Filtrage Qualitatif** (Longueur et Ratio)

---


Pour les langues à très faibles ressources, la qualité des alignements prime sur la quantité
On élimine les phrases trop longues (souvent limitées à 80 mots dans la littérature) et celles dont le ratio de longueur est disproportionn

In [3]:
# 1. Suppression des doublons
df = df.drop_duplicates()

# 2. Calcul des longueurs
df['src_len'] = df['Ruund'].apply(lambda x: len(x.split()))
df['trg_len'] = df['French'].apply(lambda x: len(x.split()))

# 3. Filtrage par longueur (max 80 tokens)
df = df[(df['src_len'] <= 80) & (df['trg_len'] <= 80)]

# 4. Filtrage par ratio de longueur (seuil de 1.5 à 2.0)
# On élimine si une phrase est 2x plus longue que sa traduction
df = df[df.apply(lambda x: max(x['src_len'], x['trg_len']) /
                         max(1, min(x['src_len'], x['trg_len'])) <= 2.0, axis=1)]

print(f"Paires restantes après filtrage : {len(df)}")

Paires restantes après filtrage : 8093


**Segmentation en sous-mots**

---


Comme le ruund est une langue morphologiquement riche, la segmentation en sous-mots est indispensable pour limiter l'explosion du vocabulaire et gérer les mots inconnus
On utilise `SentencePiece` avec une taille de vocabulaire réduite adaptée aux faibles ressources

In [4]:
import sentencepiece as spm

# Sauvegarde temporaire du texte pour l'entraînement du tokenizer
with open('data.txt', 'w', encoding='utf-8') as f:
    for t in df['Ruund'].tolist() + df['French'].tolist():
        f.write(t + '\n')

# Entraînement du modèle BPE/Unigram
spm.SentencePieceTrainer.train(
    input='data.txt',
    model_prefix='ruund_fr_sp',
    vocab_size=4000,
    model_type='bpe',
    character_coverage=1.0
)

sp = spm.SentencePieceProcessor(model_file='ruund_fr_sp.model')
df['Ruund_subwords'] = df['Ruund'].apply(lambda x: " ".join(sp.encode(x, out_type=str)))

**Division du corpus et Export**

---
La répartition standard pour l'évaluation est de 80% pour l'entraînement, 10% pour la validation et 10% pour le test final


In [5]:
from sklearn.model_selection import train_test_split

# Division 80/10/10
train, test = train_test_split(df, test_size=0.2, random_state=42)
val, test = train_test_split(test, test_size=0.5, random_state=42)

# Exportation des fichiers pour l'entraînement NMT
train[['Ruund', 'French']].to_csv('train.tsv', sep='\t', index=False)
val[['Ruund', 'French']].to_csv('val.tsv', sep='\t', index=False)
test[['Ruund', 'French']].to_csv('test.tsv', sep='\t', index=False)

print("Dataset prêt pour l'entraînement.")

Dataset prêt pour l'entraînement.


Justification des choix techniques :

---
**Normalisation NFD** : Indispensable pour les langues agglutinantes bantoues afin d'assurer que le modèle apprenne des racines cohérentes malgré les variations de tons

**Filtrage strict** : Avec un petit volume `< 10 000 paires`, l'élimination du "bruit" est la méthode la plus efficace pour améliorer le **score BLEU**

**BPE 4k **: Une petite taille de vocabulaire évite que les données ne soient trop diluées, ce qui est critique pour la convergence des modèles Transformer sur de faibles ressources
